# EDA inicial - Spotify Tracks

Esta primera parte explora la estructura y la calidad general del conjunto de datos. No incluye transformaciones avanzadas ni modelamiento.

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

## 1. Carga de los datos

In [ ]:
DATA_PATH = Path("../data/raw/Spotify_Tracks_Dataset.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(f"No se encontro el archivo: {DATA_PATH.resolve()}")

df = pd.read_csv(DATA_PATH)
print(f"Filas: {df.shape[0]:,}")
print(f"Columnas: {df.shape[1]}")
df.head()

## 2. Estructura y tipos de datos

In [ ]:
resumen_columnas = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "valores_no_nulos": df.notna().sum(),
    "valores_unicos": df.nunique(dropna=True)
})
resumen_columnas

## 3. Calidad de datos

In [ ]:
calidad = pd.DataFrame({
    "nulos": df.isna().sum(),
    "porcentaje_nulos": (df.isna().mean() * 100).round(3),
})
calidad[calidad["nulos"] > 0]

In [ ]:
print(f"Filas duplicadas completas: {df.duplicated().sum():,}")
print(f"IDs de canciones repetidos: {df['track_id'].duplicated().sum():,}")
print(f"Generos distintos: {df['track_genre'].nunique()}")
print(f"Canciones explicitas: {df['explicit'].mean() * 100:.2f}%")

**Observacion:** `Unnamed: 0` parece ser un indice guardado durante una exportacion. Para este EDA se crea una copia sin esa columna, conservando intacto el archivo original.

In [ ]:
df_eda = df.drop(columns=["Unnamed: 0"], errors="ignore").copy()
df_eda.shape

## 4. Estadisticas descriptivas

In [ ]:
variables_principales = [
    "popularity", "duration_ms", "danceability", "energy",
    "loudness", "acousticness", "valence", "tempo"
]
df_eda[variables_principales].describe().round(2)

## 5. Visualizaciones simples

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

sns.histplot(data=df_eda, x="popularity", bins=20, ax=axes[0], color="#1DB954")
axes[0].set_title("Distribucion de popularidad")
axes[0].set_xlabel("Popularidad")

duration_minutes = df_eda["duration_ms"] / 60000
sns.boxplot(x=duration_minutes, ax=axes[1], color="#1DB954")
axes[1].set_title("Duracion de las canciones")
axes[1].set_xlabel("Minutos")
axes[1].set_xlim(0, duration_minutes.quantile(0.99))

plt.tight_layout()
plt.show()

## 6. Conclusiones iniciales

- El dataset tiene 114.000 registros y 21 columnas.
- Hay solamente 3 valores nulos, distribuidos en `artists`, `album_name` y `track_name`.
- No hay filas completamente duplicadas, pero existen IDs de canciones repetidos.
- La variable `Unnamed: 0` es un indice exportado y se excluye de la copia usada para analizar.
- La duracion presenta valores extremos, incluidos registros de cero milisegundos.
- En la siguiente etapa se deben estudiar duplicados por `track_id`, valores extremos y reglas de limpieza.